# Unsupervised Full-SVD Chiral-Basis Solution of the Kitaev-Chain Spectral Problem

## I. Introduction

The chiral-basis surrogate developed elsewhere in this project
(`chiral-pinn-experiment.ipynb`) learns only the *smallest* singular triple
$(u, v)$ of the reduced $N \times N$ block $h(\mu)$. Inside the topological
phase the two near-zero singular values are split by only
$\lambda_1 \sim \mathrm{e}^{-N/\xi}$, so that model pins the near-zero
subspace and the balanced energy eigenstate but not the individual
end-localised Majorana modes, their signed per-mode wavefunctions, or the
rest of the spectrum.

Here the network instead learns the **entire** singular value decomposition

$$
h(\mu) \;=\; U(\mu)\,\Sigma(\mu)\,V(\mu)^{\mathsf T},
$$

with $U \in \mathrm{SO}(N)$ and $V \in \mathrm{O}(N)$ orthogonal by
construction and $\Sigma \ge 0$ by construction. The label-free objective
collapses to a single Frobenius residual with no folded-spectrum floor, and
one forward pass yields the whole $2N$ Bogoliubov--de Gennes spectrum, the
individual left/right Majoranas as smooth signed curves, and the
$\mathbb{Z}_2$ datum $\operatorname{sign}\det h(\mu)$.

## II. Formalism

### A. Bogoliubov--de Gennes Hamiltonian and its symmetries

In the Nambu basis $\Psi = (c_1,\dots,c_N,c_1^\dagger,\dots,c_N^\dagger)^{\mathsf T}$
the BdG Hamiltonian of the open Kitaev chain is a real symmetric
$2N \times 2N$ matrix with particle--hole symmetry
$\Xi\,H(\mu)\,\Xi = -H(\mu)$, $\Xi = \tau_x \otimes \mathbb{1}_N$.

### B. The Majorana basis

The fixed unitary $\Omega$ rotates to sublattice-ordered Majorana operators,
in which $\Omega\,H(\mu)\,\Omega^\dagger = \mathrm{i}\,M(\mu)$ with $M$ real
antisymmetric.

### C. Chiral reduction

The chiral symmetry of class BDI forces the sublattice-diagonal blocks of
$M(\mu)$ to vanish, leaving one real $N \times N$ **bidiagonal** block
$h(\mu)$: diagonal $-\mu$, super-diagonal $-(t+\Delta)$, sub-diagonal
$-(t-\Delta)$. Its singular values are exactly the non-negative BdG
eigenvalues, so $\operatorname{spec} H(\mu) = \{\pm\sigma_k(h(\mu))\}$.

### D. The full decomposition and the determinant sign

The network emits two orthogonal frames and a non-negative spectrum. Because
$\det h = \det U \cdot \prod_k \sigma_k \cdot \det V$ and both frames would
otherwise lie in $\mathrm{SO}(N)$ with $\sigma_k \ge 0$, only $\det h \ge 0$
would be representable. The continuant recurrence
$D_n = -\mu\,D_{n-1} - (t^2-\Delta^2)\,D_{n-2}$ is a Chebyshev polynomial of
the second kind, so $\det h(\mu)$ changes sign $\sim N/2$ times across
$|\mu| < 2\sqrt{t^2-\Delta^2}$ — most of the topological phase. The model
multiplies the last column of $V$ by the analytic
$s(\mu) = \operatorname{sign}\det h(\mu)$ (an $O(N)$ recurrence, detached),
restoring $V \in \mathrm{O}(N)$ and making every $\mu$ representable. Full
derivation: `docs/markdown/derivations/chiral-svd-determinant-and-fold.md`.

### E. Reflection symmetry in $\mu$

$h(-\mu) = -D\,h(\mu)\,D$ with $D = \operatorname{diag}((-1)^n)$ gives
$U(-\mu) = -D\,U(\mu)$, $V(-\mu) = D\,V(\mu)$, $\Sigma(-\mu) = \Sigma(\mu)$
column-wise. Only $|\mu|$ is fed to the backbone; training needs
$\mu \ge 0$. $N$ is even, so $s(-\mu) = s(\mu)$ and the fold is consistent.

## III. Network architecture

The surrogate is a sinusoidal representation network (SIREN). A shared
backbone maps $|\mu| / 4t$ to a feature vector, from which linear heads
produce the $N(N-1)/2$ strictly-upper entries of a skew generator for $U$
and the same for $V$:

$$
U = \exp\!\big(\mathrm{skew}(\text{head}_U)\big),\quad
V = \exp\!\big(\mathrm{skew}(\text{head}_V)\big)\cdot R(\mu),
$$

with $R(\mu) = \operatorname{diag}(1,\dots,1,s(\mu))$. Orthogonality of
$U, V$, the $\pm E$ pairing, the $\Xi$ partner and the $\mu$-fold are all
structural. $\Sigma$ is returned unsorted; the adapter selects the smallest
triple by $\arg\min_k \sigma_k$.

**Singular-value source.** `sigma_source="head"` (default) adds a third head
$\Sigma = \operatorname{softplus}(\text{head}_\Sigma)$. But $\operatorname{softplus}$
has slope $\approx \sigma$ near zero, so a head cannot move an
exponentially small $\sigma_1$ -- it is effectively frozen.
`sigma_source="rayleigh"` instead reads $\sigma_k = |u_k^{\mathsf T} h(\mu)\, v_k|$
straight off the frames: no head regresses a near-zero number, only the two
$\mathcal{O}(1)$ orthogonal frames are learned, and the Frobenius residual
reduces to $\lVert\operatorname{offdiag}(U^{\mathsf T} h V)\rVert_{\mathrm F}$
plus a soft push for $u_k^{\mathsf T} h v_k \ge 0$. The model runs CPU-only
(`torch.matrix_exp` has no MPS kernel), so `--float64` training is nearly
free and lifts the achievable $\sigma_{\min}$ floor by several orders.


## IV. Loss functional

Training minimises the single label-free Frobenius residual

$$
\mathcal{L} \;=\;
\Big\langle \big\lVert\, h(\mu)\,V \;-\; U\,\operatorname{diag}(\Sigma)\, \big\rVert_{\mathrm F}^2 \Big\rangle,
$$

with column $k$ equal to $h(\mu)\,v_k - \sigma_k\,u_k$. There is no
folded-spectrum shift (all $\sigma_k$ are fit, not just the smallest) and no
eigenvector-consistency term (a matched orthonormal frame is structural), so
$\mathcal{L}$ has no $\langle\lambda_1^2\rangle$ floor and reaches zero when
the frames are exact. An optional finite-difference frame-smoothness term
(`gauge_weight`, default $0$) makes the frame reproducible across seeds;
every physical observable is gauge-invariant, so the default is off.

**Conditioning the smallest triple.** Column $k$ enters $\mathcal{L}$ with a
scale set by $\sigma_k$: near a gap closing the topological triple has
$\sigma_1 \sim \mathrm{e}^{-N/\xi}$, so its residual contributes
$\mathcal{O}(\sigma_1^2)$ to the Frobenius norm and is swamped by the
$\mathcal{O}(t)$ bulk columns. Under a shared-step optimiser the smallest
triple is then gradient-starved, and $\sigma_{\min}(\mu)$ plateaus above its
true value while the bulk spectrum keeps improving. `ChiralSVDLoss` exposes
two default-off options that address this and compose:

- `reweight_eps` $= \varepsilon$: divide column $k$'s squared residual by
  $\sigma_k^2 + \varepsilon^2$ (the weight is detached, so it cannot be gamed
  by inflating $\sigma_k$). This is $\approx \sigma_k^{-2}$ for
  $\sigma_k \gg \varepsilon$ and saturates at $\approx \varepsilon^{-2}$ for
  $\sigma_k \ll \varepsilon$, so every triple contributes comparably with the
  near-zero column's up-weighting bounded. The smooth form has no kink, so it
  does not destabilise the L-BFGS phase. A mild folded-spectrum flavour, far
  gentler than the `ChiralFSMLoss` shift.
- `curriculum_triples` $= K$: for `curriculum_hold` epochs the $K$ smallest
  predicted triples carry weight `curriculum_start` ($0$ excludes them), so
  the well-conditioned bulk frame is fit first; the weight then rises
  linearly to $1$ over `curriculum_ramp` epochs. Keyed off the epoch
  counter, exactly as the annealing schedules of the other losses in the
  study.

The reference run below uses neither ($\varepsilon$ off, $K = 0$): the plain
residual is the baseline, and the two options are the lever if
$\sigma_{\min}$ is found to plateau above float precision.


## V. Collocation sampling

Collocation points are drawn from $\mu \in [\delta,\,4t]$, $\delta = 0.05\,t$,
with the half-domain region mixture of `four_model_comparison.py`. The
neighbourhood of the origin is excluded: the $-D$ factor of the fold is
sign-discontinuous at $\mu = 0$, a measure-zero gauge artefact. The
experimental run below redraws fresh batches at every optimiser step
(`mode="infinite"`); frozen pools are used for the validation curve and the
L-BFGS stage.

## VI. Implementation

### A. Imports

The model, loss, sampling, probes, and training components used below are
provided by the `kitaev` package.

In [ ]:
%matplotlib inline
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from accelerate import Accelerator
from sesh import Session

from kitaev.analytical import KitaevChainHamiltonian, chiral_block
from kitaev.data.sampling_region import SamplingRegion
from kitaev.models import ChiralFullToBdGAdapter, SirenPINNChiralFull
from kitaev.training import (
    BdGEvaluationProbe,
    SpectrumEvaluationProbe,
    TwoPhaseConfig,
    run_two_phase,
)
from kitaev.training.config import TrainerConfig
from kitaev.training.loss import ChiralSVDLoss
from kitaev.training.sampling import SamplingConfig, build_sampling
from kitaev.training.trainer import _build_kitaev_operators
from kitaev.visualisation import mark_transition, use_house_style

### B. Session and device

`torch.matrix_exp` has no MPS kernel, so on Apple Silicon the run falls back
to CPU; CUDA is unaffected.

In [ ]:
session = Session(
    name="chiral-svd-experiment",
    output_root=Path("../../results/logs"),
    enable_mlflow=False,
)

force_cpu = torch.backends.mps.is_available() and not torch.cuda.is_available()
accelerator = Accelerator(cpu=force_cpu)
session.info(f"Using device: {accelerator.device}")

### C. Hamiltonian

All quantities are in units of $t$, with $N = 20$ and $\Delta = 0.5\,t$, so
the transition lies at $|\mu| = 2t$. `ChiralSVDLoss` rebuilds $h(\mu)$ from
the batch; the $(H_{\mathrm{base}}, H_{\mu\text{-diag}}, \Xi)$ triple is
passed only for the trainer's interface.

In [ ]:
N = 20
t = 1.0
delta = 0.5
session.info(f"N = {N}, t = {t}, delta = {delta}, transition at |mu| = {2 * t}")

hamiltonian = KitaevChainHamiltonian(n_sites=N, hopping=t, pairing=delta)

H_base, H_mu_diag, Xi = _build_kitaev_operators(N, hopping=t, pairing=delta)
H_base = H_base.to(accelerator.device)
H_mu_diag = H_mu_diag.to(accelerator.device)
Xi = Xi.to(accelerator.device)

### D. Collocation sampling

The half-domain region mixture is passed to `build_sampling`. Two further
`mode="frozen"` pools are built: a small held-out set for the validation
curve, and a large single-batch pool for the L-BFGS stage (a quasi-Newton
method needs a stationary objective).

In [ ]:
torch.manual_seed(42)

HALF_REGIONS = (
    SamplingRegion(low=0.05, high=4.0, weight=1.0),
    SamplingRegion(low=1.7, high=2.6, weight=1.5),
    SamplingRegion(low=2.0, high=4.0, weight=0.5),
)

train_loader, sampling_callbacks = build_sampling(
    SamplingConfig(mode="infinite", batch_size=1024, steps_per_epoch=8),
    HALF_REGIONS,
)
val_loader, _ = build_sampling(
    SamplingConfig(mode="frozen", batch_size=1024, total_samples=1024),
    HALF_REGIONS,
)
lbfgs_loader, _ = build_sampling(
    SamplingConfig(mode="frozen", batch_size=2048, total_samples=2048),
    HALF_REGIONS,
)

### E. Network, loss, optimisation schedule, and error probes

The backbone is `hidden_features = 64`, `hidden_layers = 2`; the input scale
$4.0$ matches $[0, 4t]$. `ChiralSVDLoss` carries no schedule, `gauge_weight
= 0`, and the smallest-triple conditioning options of Sec. IV
(`reweight_eps`, `curriculum_triples`) at their defaults (off). The model
below uses `sigma_source="head"`; `"rayleigh"` (Sec. III) and `--float64`
are the levers when the near-zero sector is the bottleneck. Two probes run
against exact diagonalisation on a fixed grid: `BdGEvaluationProbe` scores
the lowest eigenpair through `ChiralFullToBdGAdapter` (the same metrics as
every other model in the study), and `SpectrumEvaluationProbe` scores the
whole $2N$ spectrum.


In [ ]:
model = SirenPINNChiralFull(
    n_sites=N,
    hidden_features=64,
    hidden_layers=2,
    input_scale=4.0,
    hopping=t,
    pairing=delta,
)

# reweight_eps / curriculum_triples default-off; see Sec. IV for the lever.
loss_fn = ChiralSVDLoss(n_sites=N, hopping=t, pairing=delta, gauge_weight=0.0)

TWO_PHASE = TwoPhaseConfig(
    adam_epochs=3000,
    adam_lr=8e-4,
    adam_weight_decay=1e-6,
    lbfgs_epochs=100,
    lbfgs_max_iter=20,
    lbfgs_history_size=20,
    lbfgs_line_search_fn="strong_wolfe",
)
two_phase_base = TrainerConfig(
    epochs=1, print_freq=500, patience=None, grad_clip_norm=1.0
)

MU_GRID = np.linspace(-4.0, 4.0, 240)


def adapt(m):
    return ChiralFullToBdGAdapter(m, hopping=t, pairing=delta)


probe = BdGEvaluationProbe(
    n_sites=N,
    hopping=t,
    pairing=delta,
    mu_grid=MU_GRID,
    every=50,
    session=session,
    adapt=adapt,
)
spectrum_probe = SpectrumEvaluationProbe(
    n_sites=N,
    spectrum=lambda m, x: adapt(m).full_spectrum(x),
    hopping=t,
    pairing=delta,
    mu_grid=MU_GRID,
    every=50,
    session=session,
)

### F. Training

Two stages on a single network: AdamW under a cosine decay on the streaming
loader, then L-BFGS under a strong-Wolfe line search on the fixed pool. This
is a full run; reduce `TWO_PHASE.adam_epochs` for a quick pass.

In [ ]:
trained_model, history = run_two_phase(
    session=session,
    accelerator=accelerator,
    model=model,
    loss_fn=loss_fn,
    train_loader=train_loader,
    H_base=H_base,
    H_mu_diag=H_mu_diag,
    Xi=Xi,
    two_phase=TWO_PHASE,
    base_config=two_phase_base,
    callbacks=[*sampling_callbacks, probe, spectrum_probe],
    val_loader=val_loader,
    lbfgs_train_loader=lbfgs_loader,
    lbfgs_callbacks=[probe, spectrum_probe],
)

adapter = ChiralFullToBdGAdapter(trained_model, hopping=t, pairing=delta).to(
    accelerator.device
)
adapter.eval()
session.info(
    f"two-phase run complete: {len(history['train_loss'])} epochs "
    f"(train svd {history['train_svd'][-1]:.3e})"
)

## VII. Results

All comparisons are against exact diagonalisation of $H(\mu)$ on a uniform
grid. The adapter exposes the smallest triple as $(E_{\mathrm{pred}} \ge 0,
\psi_{\mathrm{pred}})$ and adds `full_spectrum` and `det_sign`.

### Figure style

The topological region $|\mu| < 2t$ is shaded and the transition at
$|\mu| = 2t$ marked on every figure below.

In [ ]:
use_house_style()

MU_SWEEP = np.linspace(-4.0, 4.0, 400)
mu_pos = MU_SWEEP[MU_SWEEP >= 0]

### A. Optimisation history and physical error

The Frobenius residual `train_svd` and the probe series
(`probe_e_mae`, `probe_subspace_infidelity`, `probe_spectrum_mae`) are
plotted against the cumulative epoch index, with the AdamW $\to$ L-BFGS
hand-over marked. Unlike the single-triple loss, `train_svd` has no
finite-size floor and should fall towards float precision.

In [ ]:
epochs = range(1, len(history["train_loss"]) + 1)
split = TWO_PHASE.adam_epochs

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
axes[0].plot(epochs, history["train_svd"], color="#1b2a41", lw=1.8, label="train")
if "val_svd" in history:
    axes[0].plot(
        epochs, history["val_svd"], color="#e4572e", lw=1.3, ls=(0, (3, 2)), label="val"
    )
axes[0].axvline(split, color="#8d99ae", ls=(0, (2, 2)), lw=1.0)
axes[0].set_yscale("log")
axes[0].set_title("Frobenius residual")
axes[0].set_xlabel("epoch")
axes[0].legend()

pe = history["probe_spectrum_epoch"]
axes[1].plot(
    pe, history["probe_spectrum_mae"], color="#1b2a41", lw=1.6, label="spectrum MAE"
)
axes[1].plot(
    pe, history["probe_e_mae"], color="#e4572e", lw=1.6, label="lowest-|E| MAE"
)
axes[1].set_yscale("log")
axes[1].set_title("Probe errors vs exact diagonalisation")
axes[1].set_xlabel("evaluation index")
axes[1].legend()
fig.tight_layout()
fig.savefig(session.path() / "losses.png", bbox_inches="tight")

### B. The full $2N$ spectrum

Every predicted level $\pm\sigma_k(\mu)$ over exact diagonalisation, with a
residual panel. The tracking should hold for the whole spectrum, not only
the lowest level.

In [ ]:
ham = hamiltonian
exact = np.array([np.sort(np.linalg.eigvalsh(ham.build(float(m)))) for m in MU_SWEEP])
with torch.no_grad():
    pred = (
        adapter.full_spectrum(
            torch.tensor(
                MU_SWEEP[:, None], dtype=torch.float32, device=accelerator.device
            )
        )
        .cpu()
        .numpy()
    )

fig, (ax_top, ax_bot) = plt.subplots(
    2, 1, figsize=(8, 6), height_ratios=(3, 1), sharex=True
)
for k in range(2 * N):
    ax_top.plot(MU_SWEEP, exact[:, k], color="0.7", lw=2.0)
    ax_top.plot(MU_SWEEP, pred[:, k], color="#e4572e", lw=0.9)
ax_top.set_ylabel(r"BdG eigenvalue $E$")
ax_top.set_title("Full spectrum: predicted (coral) over exact (grey)")
mark_transition(ax_top, hopping=t, mu_max=4.0, two_sided=True)
ax_bot.plot(MU_SWEEP, np.abs(pred - exact).max(axis=1), color="#1b2a41", lw=1.2)
ax_bot.set_yscale("log")
ax_bot.set_ylabel("max level error")
ax_bot.set_xlabel(r"$\mu / t$")
mark_transition(ax_bot, hopping=t, mu_max=4.0, two_sided=True)
fig.tight_layout()
fig.savefig(session.path() / "2N_spectrum.png", bbox_inches="tight")

### C. Eigenvector fidelity and edge localisation

$\lVert P\,\psi_{\mathrm{pred}} \rVert$ onto the span of the two exact
smallest-$|E|$ eigenvectors (unity when $\psi_{\mathrm{pred}}$ lies in that
subspace), and the combined particle+hole weight on the two outermost sites
at each end.

In [ ]:
n_edge = 2
edge_idx = np.r_[0:n_edge, N - n_edge : N]
with torch.no_grad():
    _e, psi_pred_t = adapter(
        torch.tensor(mu_pos[:, None], dtype=torch.float32, device=accelerator.device)
    )
psi_pred = psi_pred_t.cpu().numpy()
psi_pred = psi_pred / np.linalg.norm(psi_pred, axis=1, keepdims=True)

fidelity = np.empty_like(mu_pos)
edge_exact = np.empty_like(mu_pos)
for i, m in enumerate(mu_pos):
    w, V = np.linalg.eigh(ham.build(float(m)))
    near = V[:, np.argsort(np.abs(w))[:2]]
    fidelity[i] = np.linalg.norm(near.T @ psi_pred[i])
    pe_ = V[:, N]
    edge_exact[i] = (pe_[:N][edge_idx] ** 2).sum() + (pe_[N:][edge_idx] ** 2).sum()
edge_pred = (psi_pred[:, :N][:, edge_idx] ** 2).sum(1) + (
    psi_pred[:, N:][:, edge_idx] ** 2
).sum(1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for ax in axes:
    mark_transition(ax, hopping=t, mu_max=4.0, two_sided=False)
axes[0].plot(mu_pos, fidelity, color="#1b2a41", lw=2.0)
axes[0].set_title(r"subspace fidelity $\|P\,\psi_{\mathrm{pred}}\|$")
axes[0].set_xlabel(r"$\mu / t$")
axes[1].plot(mu_pos, edge_exact, color="#1b2a41", lw=2.0, label="exact")
axes[1].plot(mu_pos, edge_pred, color="#e4572e", lw=1.6, ls=(0, (3, 2)), label="PINN")
axes[1].set_title("edge weight")
axes[1].set_xlabel(r"$\mu / t$")
axes[1].legend()
fig.tight_layout()
fig.savefig(session.path() / "fidelity.png", bbox_inches="tight")

### D. Probability densities

Particle- and hole-sector densities $|\psi_n|^2$ at representative $\mu$: two
deep in the topological phase, one near the transition, two trivial.

In [ ]:
probe_mus = [0.3, 1.0, 1.9, 2.6, 3.4]
sites = np.arange(N)
fig, axes = plt.subplots(
    2, len(probe_mus), figsize=(3.0 * len(probe_mus), 5.2), sharex=True, sharey="row"
)
for col, mu in enumerate(probe_mus):
    with torch.no_grad():
        _e, psi_t = adapter(
            torch.tensor([[mu]], dtype=torch.float32, device=accelerator.device)
        )
    psi = psi_t.cpu().numpy().ravel()
    psi = psi / np.linalg.norm(psi)
    w, V = np.linalg.eigh(ham.build(float(mu)))
    ex = V[:, N]
    for row, (lo, hi, label) in enumerate(((0, N, "particle"), (N, 2 * N, "hole"))):
        ax = axes[row, col]
        ax.fill_between(sites, ex[lo:hi] ** 2, color="#1b2a41", alpha=0.12, lw=0)
        ax.plot(sites, psi[lo:hi] ** 2, color="#e4572e", lw=1.5, ls=(0, (3, 2)))
        if col == 0:
            ax.set_ylabel(label)
    axes[0, col].set_title(rf"$\mu = {mu:.1f}\,t$")
    axes[-1, col].set_xlabel("site $n$")
fig.tight_layout()
fig.savefig(session.path() / "probdense.png", bbox_inches="tight")

### E. Reflection symmetry

The energy at $-\mu$ via the internal fold $U \to -D U$, $V \to D V$ compared
with the energy at $\mu$; they agree to floating-point tolerance by
construction.

In [ ]:
with torch.no_grad():
    e_pos = (
        adapter(
            torch.tensor(
                mu_pos[:, None], dtype=torch.float32, device=accelerator.device
            )
        )[0]
        .cpu()
        .numpy()
        .ravel()
    )
    e_neg = (
        adapter(
            torch.tensor(
                -mu_pos[:, None], dtype=torch.float32, device=accelerator.device
            )
        )[0]
        .cpu()
        .numpy()
        .ravel()
    )

fig, ax = plt.subplots(figsize=(8, 3.8))
mark_transition(ax, hopping=t, mu_max=4.0, two_sided=True)
ax.plot(mu_pos, e_pos, color="#1b2a41", lw=2.0, label=r"$E(\mu)$")
ax.plot(
    -mu_pos, e_neg, color="#e4572e", lw=1.6, ls=(0, (4, 2)), label=r"$E(-\mu)$ via fold"
)
ax.set_xlabel(r"$\mu / t$")
ax.set_ylabel("$E$")
ax.legend()
fig.tight_layout()
fig.savefig(session.path() / "reflection.png", bbox_inches="tight")
session.info(f"max |E(-mu) - E(mu)| via fold: {np.abs(e_neg - e_pos).max():.2e}")

### F. Individual left/right Majorana modes

In the chiral SVD the near-zero pair is the *single* smallest triple
$(u_1, v_1)$ -- the second singular value is $\mathcal{O}(t)$, not a second
near-zero mode. Its left and right singular vectors are themselves the two
end-localised Majoranas: $\psi_{\text{left}}$ has site density
$|u_1(n)|^2$ and $\psi_{\text{right}}$ has $|v_1(n)|^2$. These are compared
with the smallest singular vectors of the exact chiral block at several
topological $\mu$; they should localise on opposite ends and sharpen as
$\mu \to 0$.


In [ ]:
majorana_mus = [0.3, 0.9, 1.5, 1.9]
fig, axes = plt.subplots(
    1, len(majorana_mus), figsize=(3.2 * len(majorana_mus), 3.0), sharey=True
)
for ax, mu in zip(axes, majorana_mus, strict=True):
    x = torch.tensor([[mu]], dtype=torch.float32, device=accelerator.device)
    with torch.no_grad():
        u_mat, sigma, v_mat = trained_model(x)
    k = int(torch.argmin(sigma[0]))
    pred_u = (u_mat[0, :, k] ** 2).cpu().numpy()
    pred_v = (v_mat[0, :, k] ** 2).cpu().numpy()

    block = chiral_block(mu, N, t, delta)
    u_ex, s_ex, vt_ex = np.linalg.svd(block)
    j = int(np.argmin(s_ex))
    exact_l, exact_r = u_ex[:, j] ** 2, vt_ex[j, :] ** 2

    # Match the predicted pair to the exact pair, then orient left-first.
    if pred_u @ exact_l + pred_v @ exact_r < pred_u @ exact_r + pred_v @ exact_l:
        pred_u, pred_v = pred_v, pred_u
    if exact_l[: N // 2].sum() < exact_l[N // 2 :].sum():
        pred_u, pred_v, exact_l, exact_r = pred_v, pred_u, exact_r, exact_l

    ax.fill_between(sites, exact_l, color="#2a9d8f", alpha=0.25, lw=0)
    ax.fill_between(sites, exact_r, color="#e9c46a", alpha=0.25, lw=0)
    ax.plot(sites, pred_u, color="#2a9d8f", lw=1.4, label="left end mode")
    ax.plot(sites, pred_v, color="#e9c46a", lw=1.4, label="right end mode")
    ax.set_title(rf"$\mu = {mu:.2f}\,t$")
    ax.set_xlabel("site $n$")
axes[0].set_ylabel("end-mode site density (predicted lines, exact shaded)")
axes[0].legend(fontsize=8)
fig.tight_layout()
fig.savefig(session.path() / "LR-mode.png", bbox_inches="tight")

### G. The $\mathbb{Z}_2$ datum from the learned frame

$\operatorname{sign}\det h(\mu)$ read off the frame as
$\operatorname{sign}\det U \cdot \operatorname{sign}\det V$, compared with the
analytic value, and $\sigma_{\min}(\mu)$ vs exact. `det_sign` returns exactly
the $s(\mu)$ fed in, so it verifies the frame carries the datum consistently;
the learned signature of the transition at $|\mu| = 2t$ is $\sigma_{\min}$
dropping from $O(1)$ to $\sim 10^{-6}$.

In [ ]:
x_grid = torch.tensor(MU_SWEEP[:, None], dtype=torch.float32, device=accelerator.device)
with torch.no_grad():
    det_pred = adapter.det_sign(x_grid).cpu().numpy()
    _u, sigma_all, _v = trained_model(x_grid)
sigma_min = sigma_all.min(dim=1).values.cpu().numpy()
det_exact = np.array(
    [np.sign(np.linalg.det(chiral_block(m, N, t, delta))) for m in MU_SWEEP]
)
sigma_exact = np.array(
    [np.abs(np.linalg.eigvalsh(ham.build(float(m)))).min() for m in MU_SWEEP]
)

fig, (ax_top, ax_bot) = plt.subplots(2, 1, figsize=(8, 5), sharex=True)
ax_top.step(MU_SWEEP, det_exact, color="0.7", lw=2.5, where="mid", label="analytic")
ax_top.step(MU_SWEEP, det_pred, color="#e4572e", lw=1.1, where="mid", label="frame")
ax_top.set_ylabel(r"$\mathrm{sign}\,\det h$")
ax_top.set_yticks((-1, 1))
ax_top.legend(fontsize=8)
mark_transition(ax_top, hopping=t, mu_max=4.0, two_sided=True)
ax_bot.plot(MU_SWEEP, sigma_exact, color="0.7", lw=2.5, label="exact")
ax_bot.plot(MU_SWEEP, sigma_min, color="#1b2a41", lw=1.1, label="predicted")
ax_bot.set_yscale("log")
ax_bot.set_ylabel(r"$\sigma_{\min}(\mu)$")
ax_bot.set_xlabel(r"$\mu / t$")
ax_bot.legend(fontsize=8)
mark_transition(ax_bot, hopping=t, mu_max=4.0, two_sided=True)
fig.tight_layout()
session.info(f"det-sign agreement: {np.mean(det_pred == det_exact):.3f}")
fig.savefig(session.path() / "z2.png", bbox_inches="tight")

## VIII. Frame reproducibility across seeds

Two runs of this model settle on frames that differ by the residual SVD gauge
(matched column sign flips, degenerate-block rotations) even when every
physical observable agrees. The gauge-invariant per-triple site density
$u_k^2 + v_k^2$ is reproducible across seeds; the raw frame $U(\mu)$ is not,
unless a small `gauge_weight` is used. `experiments/chiral_svd_spectrum.py
--seeds 0 1 2` produces the quantitative `frame_reproducibility` panel; the
cell below sketches the comparison for a single retrained seed.

In [ ]:
# Illustrative: retrain a second seed briefly and compare the gauge-invariant
# density with the gauge-dependent frame. For the full multi-seed measurement
# use experiments/chiral_svd_spectrum.py.
torch.manual_seed(7)
model_b = SirenPINNChiralFull(
    n_sites=N,
    hidden_features=64,
    hidden_layers=2,
    input_scale=4.0,
    hopping=t,
    pairing=delta,
)
# (left unexecuted in the committed notebook; see the experiment script)

## IX. Summary

Emitting the prediction in the symmetry-adapted basis and as the *full*
decomposition turns the $2N \times 2N$ eigenproblem with a
$\lambda_1$-flat near-zero manifold into one $N \times N$ SVD with $U, V$
orthogonal and $\Sigma \ge 0$ by construction. The label-free objective is a
single Frobenius residual with no floor; one forward pass gives the whole
$2N$ spectrum, the individual left/right Majoranas as smooth signed curves,
and the $\mathbb{Z}_2$ datum from the frame. The one piece of analytic
machinery the full frame requires that the single triple did not is the
last-column reflection $s(\mu) = \operatorname{sign}\det h(\mu)$, without
which $\det h < 0$ across half the topological phase would be unrepresentable.

The bare Frobenius residual weights each triple by $\sigma_k$, so the
exponentially small topological triple is gradient-starved: the practical
signature is $\sigma_{\min}(\mu)$ plateauing above float precision while the
bulk spectrum converges. The `reweight_eps` and `curriculum_triples` options
of Sec. IV are the conditioning levers for that regime, both default-off so
the reference run is the unweighted residual. The reduction is by
representation: `SirenPINNChiral` / `ChiralFSMLoss` remain the baseline, and
`ChiralFullToBdGAdapter` scores this model on the same lowest-pair metrics as
every other model in the study.

**Reading the near-zero sector.** Below the finite-size splitting
$\lambda_1 \sim \mathrm{e}^{-N/\xi}$ the *value* of $\sigma_{\min}$ is not a
physical quantity, so the raw near-zero spectrum MAE there measures float
precision, not accuracy. The pass criteria are instead the topological
subspace infidelity, the ratio $\sigma_1/\sigma_2$ (should be
$\ll 1$), the individual-Majorana end localisation, and the exact
$\mathbb{Z}_2$ sign -- the experiment reports these alongside the MAE.
